## مختبر: بناء ومقارنة وكيل ذكي مع شات بوت تقليدي

في هذا المختبر ستتعلم الفرق بين الشات بوت البسيط والوكيل الذكي عبر بناء نموذجين عمليين:
- شات بوت قائم على قواعد محددة.
- وكيل يستخدم LangChain وأداة لتحليل البيانات.

**المتطلبات**: Python, `langchain`, `openai`, `pandas`, `numpy`.

## 1. تجهيز البيئة والمكتبات

سنتأكد من استيراد المكتبات وضبط مفتاح DeepSeek API (متاح في بيئة التشغيل).

In [ ]:
import os
import numpy as np
import pandas as pd

# التأكد من وجود مفتاح API
assert 'DEEPSEEK_API_KEY' in os.environ, 'يجب توفير DEEPSEEK_API_KEY'
print('المفتاح متاح ✅')

## 2. تحميل أو إنشاء مجموعة بيانات

سننشئ DataFrame عشوائيًا باستخدام numpy لإيضاح أداة الوكيل لاحقًا.

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    'x': np.random.randn(100),
    'y': np.random.randn(100),
    'category': np.random.choice(['A', 'B', 'C'], size=100)
})
df.head()

## 3. بناء شات بوت بسيط (قائم على القواعد)

سننشئ دالة `rule_chatbot` تستجيب فقط لبعض التحيات والأسئلة المحددة مسبقًا. هذا النوع من النماذج لا يملك القدرة على استخدام أدوات أو التعامل مع أسئلة غير متوقعة.

In [ ]:
def rule_chatbot(user_input):
    # TODO: أكمل الشروط لتشميل تحية أخرى، مثلاً 'أهلاً'
    if user_input.strip().lower() == 'مرحبا':
        return 'أهلاً بك! كيف يمكنني مساعدتك؟'
    elif user_input.strip().lower() == 'كيف حالك؟':
        return 'أنا بخير، شكراً لسؤالك!'
    elif user_input.strip() == '':
        return 'الرجاء إدخال نص.'
    else:
        return 'عذراً، لم أفهم سؤالك.'

In [ ]:
# اختبار الشات بوت
test_inputs = ['مرحبا', 'كيف حالك؟', 'ما هو متوسط العمود x؟', 'أهلاً']
for inp in test_inputs:
    print(f'المدخل: {inp} => الاستجابة: {rule_chatbot(inp)}')

## 4. بناء وكيل ذكي باستخدام LangChain وأداة مخصصة

الآن سنستخدم LangChain لبناء وكيل يمتلك أداة `data_mean` التي تحسب متوسط عمود معين من DataFrame. الوكيل سيستخدم DeepSeek للتفكير واتخاذ القرار واستدعاء الأداة عند الحاجة.

**تنبيه**: سنستخدم خادم DeepSeek API (OpenAI-compatible).

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool

# إعداد LLM
llm = ChatOpenAI(
    model='deepseek-chat',
    openai_api_key=os.environ['DEEPSEEK_API_KEY'],
    openai_api_base='https://api.deepseek.com/v1',
    temperature=0
)

# تعريف أداة مخصصة
@tool
def data_mean(column_name: str) -> float:
    """
    Calculate the mean of a column in the global DataFrame df.
    Input: column name (string)
    Returns: mean value (float)
    """
    # TODO: حساب المتوسط للعمود المطلوب وإرجاعه
    pass

# قائمة الأدوات
tools = [data_mean]

In [ ]:
# تهيئة الوكيل
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,  # لإظهار عملية التفكير
    handle_parsing_errors=True
)

## 5. اختبار الوكيل ومقارنته بالشات بوت

سنطرح على الوكيل سؤالاً يتطلب استخدام الأداة. لاحظ كيف يفكر الوكيل ويستدعي الأداة.

In [ ]:
# سؤال يتطلب حساب متوسط العمود x
query = ' ما هو متوسط العمود x في مجموعة البيانات؟'
response = agent.invoke(query)
print('\nالإجابة النهائية:', response['output'])

## 6. ملاحظة الفرق

الآن دعنا نقارن: اسأل نفس السؤال للشات بوت القواعدي.

In [ ]:
print('استجابة الشات بوت القواعدي:')
print(rule_chatbot(query))

print('\nاستجابة الوكيل:')
print(response['output'])

## 7. خلاصة

كما رأيت:
- **الشات بوت** عاجز عن فهم السؤال لأنه لا يملك قدرة التفكير أو استخدام الأدوات.
- **الوكيل الذكي** فهم السؤال، استخدم أداة `data_mean` لحساب المتوسط، وأرجع الإجابة.

هذا يوضح مكونات الوكيل: الإدراك (interpretation)، التفكير (planning)، الفعل (tool execution)، ودورة Perceive-Think-Act.